## Improving SQL generation with reflection
This notebook demonstrates how to use LLM-based reflection and feedback loops to improve the accuracy of generated SQL queries. We explore two approaches: internal reflection (where the LLM critiques its own output) and external feedback (where the LLM uses actual database results to refine the query).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import src.utils as utils
from src.agents import generate_sql, refine_sql, refine_sql_external_feedback

# Configuration
DB_PATH = 'products.db'
MODEL = "openai:gpt-4.1"

## 1. Environment and Database Setup
We start by loading environment variables and setting up a sample database with transaction data.

In [ ]:
utils.create_transactions_db(db_name=DB_PATH)
utils.print_html(utils.get_schema(DB_PATH))

## 2. Generating the Initial SQL Query (V1)
In this step, we use the database schema and a natural language question to generate our first SQL query.

In [ ]:
# Example usage of generate_sql

# We provide the schema as a string
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

# We ask a question about the data in natural language
question = "Which color of product has the highest total sales?"

utils.print_html(question, title="User Question")

# Generate the SQL query using the specified model
sql_V1 = generate_sql(question, schema, model=MODEL)

# Display the generated SQL query
utils.print_html(sql_V1, title="SQL Query V1")

## 3. Executing the Initial Query
We execute the generated SQL against our database to see the raw output. This output will serve as the baseline for our refinement steps.

In [ ]:
# Execute the generated SQL query (sql_V1) against the products.db database.
# The result is returned as a pandas DataFrame.
df_sql_V1 = utils.execute_sql(sql_V1, db_path=DB_PATH)

# Render the DataFrame as an HTML table in the notebook.
# This makes the query output easier to read and interpret.
utils.print_html(df_sql_V1, title="Output of SQL Query V1")


## 4. Approach A: Internal Reflection (V1 → V2)
In this approach, the LLM evaluates its own SQL query based on the schema and the user's question, without seeing the actual query results. It then provides feedback and generates a refined version (V2).

In [ ]:
# Example: refine the generated SQL (V1 → V2)

feedback, sql_V2 = refine_sql(
    question=question,
    sql_query=sql_V1,   # <- comes from generate_sql() (V1)
    schema=schema, # <- we reuse the schema from section 3.1
    model=MODEL
)

# Display the original prompt
utils.print_html(question, title="User Question")

# --- V1 ---
utils.print_html(sql_V1, title="Generated SQL Query (V1)")

# --- Feedback + V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

# Execute and show V2 output
df_sql_V2 = utils.execute_sql(sql_V2, db_path=DB_PATH)
utils.print_html(df_sql_V2, title="SQL Output of V2")

## 5. Approach B: Refinement with External Feedback (V1 → V3)
Here, the LLM is given both the initial SQL query and its actual execution output. This "external feedback" allows the LLM to identify errors or gaps in the results and produce a more accurate refined query (V3).

In [ ]:
# Example: Refine SQL with External Feedback (V1 → V3)

# Execute the original SQL (V1)
df_sql_V1 = utils.execute_sql(sql_V1, db_path=DB_PATH)

# Use external feedback to evaluate and refine
feedback, sql_V3 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,   # V1 query
    df_feedback=df_sql_V1,    # Output of V1
    schema=schema,
    model=MODEL
)

# --- V1 ---
utils.print_html(question, title="User Question")
utils.print_html(sql_V1, title="Generated SQL Query (V1)")
utils.print_html(df_sql_V1, title="SQL Output of V1")

# --- Feedback & V3 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V3, title="Refined SQL Query (V3)")

# Execute and display V3 results
df_sql_V3 = utils.execute_sql(sql_V3, db_path=DB_PATH)
utils.print_html(df_sql_V3, title="SQL Output of V3 (with External Feedback)")
